# IQ MOHAWK — Phi-4-mini → IQ mixer
This notebook runs the first curated transfer experiment entirely on a Colab GPU. It does **not** require a local GPU.

The experiment is intentionally narrow: Phi-4-mini softmax attention → IQ linear-time mixer, using MOHAWK Stage 1 matrix orientation.

In [ ]:
import torch, subprocess, sys
assert torch.cuda.is_available(), 'Enable a GPU: Runtime > Change runtime type > GPU'
print(torch.cuda.get_device_name(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())
subprocess.run(['nvidia-smi'], check=False)

In [ ]:
!pip -q install 'transformers==4.49.0' 'accelerate==1.3.0' safetensors huggingface_hub
!rm -rf /content/IQ
!git clone --depth 1 --branch experiment/mohawk-phi4-mini-iq https://github.com/Harqer/IQ.git /content/IQ
%cd /content/IQ

In [ ]:
!python -m unittest tests.test_mohawk -v

## Stage 1 smoke run
The default is small enough to test the transfer pipeline before spending a long Colab session. The held-out metric matters; training loss alone is not a pass.

In [ ]:
!python -m experiments.mohawk_phi4_mini.stage1 \
  --teacher microsoft/Phi-4-mini-instruct \
  --corpus-root /content/IQ \
  --layer 15 \
  --seq-len 192 \
  --train-steps 32 \
  --eval-chunks 8 \
  --output /content/IQ/artifacts/mohawk_phi4_mini_stage1.pt

In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path('/content/IQ/artifacts/mohawk_phi4_mini_stage1.json').read_text())
metrics

## Persist artifacts to Google Drive (optional)
Colab runtimes are ephemeral. Mount Drive only if you want the checkpoint and metrics copied out of the runtime.

In [ ]:
# Uncomment after connecting Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p '/content/drive/MyDrive/IQ/mohawk_phi4_mini'
# !cp /content/IQ/artifacts/mohawk_phi4_mini_stage1.* '/content/drive/MyDrive/IQ/mohawk_phi4_mini/'